# Voting Classifier

- We use the Breast Cancer dataset to predict whether a tumor is:
 - Malignant (0)
 - Benign (1)
- This is a binary classification problem.

## Import Required Libraries

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

## Load & Split the data

In [2]:
data = load_breast_cancer()
X = data.data
y = data.target

In [6]:
print(data)

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]]), 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
 

In [3]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, y, test_size=0.25, random_state=13)

## Define Base Models

In [4]:
# probabilistic model
model1 = GaussianNB()

# distance-based model
model2 = KNeighborsClassifier(n_neighbors=5)

# shallow tree
model3 = DecisionTreeClassifier(max_depth=3, random_state=42)

## Train Individual Models & Compare Accuracy

In [8]:
# Train Gaussian Naive Bayes
model1.fit(Xc_train, yc_train)
acc_gnb = model1.score(Xc_test, yc_test)

# Train KNN Classifier
model2.fit(Xc_train, yc_train)
acc_knn = model2.score(Xc_test, yc_test)

# Train Decision Tree Classifier
model3.fit(Xc_train, yc_train)
acc_dt = model3.score(Xc_test, yc_test)

# Display results
pd.DataFrame(
    {
        "Model": ["GaussianNB", "KNN", "Decision Tree"],
        "Accuracy": [acc_gnb, acc_knn, acc_dt]
    }
)

,Model,Accuracy
0,GaussianNB,0.930070
1,KNN,0.923077
2,Decision Tree,0.930070


## Voting Classifier (Hard Voting)

Why (name, model) is required instead of just [model1, model2, model3]

- In VotingClassifier, scikit-learn requires tuples of (name, estimator) because the names are used internally for several purposes.

In [9]:
# The VotingClassifier combines predictions from all base models using majority voting.

voting_clf = VotingClassifier(
    estimators=[
        ('gnb', model1),
        ('knn', model2),
        ('dt', model3)
    ],
    voting='hard'
)

voting_clf.fit(Xc_train, yc_train)

y_pred = voting_clf.predict(Xc_test)
voting_acc = accuracy_score(yc_test, y_pred)

print("Voting Classifier Accuracy:", voting_acc)

Voting Classifier Accuracy: 0.9440559440559441


- Since individual models make different errors, hard voting selects the majority class, often resulting in slightly better performance than any single model.

In [13]:
voting_clf.named_estimators_['knn']

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [14]:
print(voting_clf.estimators)

[('gnb', GaussianNB()), ('knn', KNeighborsClassifier()), ('dt', DecisionTreeClassifier(max_depth=3, random_state=42))]


# Voting Regressor

- We use the California Housing dataset to predict the median house value.This is a regression problem with continuous output.

In [12]:
from sklearn.datasets import fetch_california_housing

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.ensemble import VotingRegressor
from sklearn.metrics import mean_squared_error

## Load and Split the Dataset

In [15]:
housing = fetch_california_housing()

In [16]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)

## Define Base Regression Models

In [17]:
reg1 = LinearRegression()
reg2 = DecisionTreeRegressor(random_state=42)
reg3 = KNeighborsRegressor()

## Train Individual Regressors & Compare MSE

In [18]:
# Train Linear Regression
reg1.fit(Xr_train, yr_train)
y_pred_lr = reg1.predict(Xr_test)
mse_lr = mean_squared_error(yr_test, y_pred_lr)

# Train Decision Tree Regressor
reg2.fit(Xr_train, yr_train)
y_pred_dt = reg2.predict(Xr_test)
mse_dt = mean_squared_error(yr_test, y_pred_dt)

# Train KNN Regressor
reg3.fit(Xr_train, yr_train)
y_pred_knn = reg3.predict(Xr_test)
mse_knn = mean_squared_error(yr_test, y_pred_knn)

# Display results
pd.DataFrame(
    {
        "Model": ["Linear Regression", "Decision Tree", "KNN"],
        "MSE": [mse_lr, mse_dt, mse_knn]
    }
)

,Model,MSE
0,Linear Regression,0.555892
1,Decision Tree,0.499707
2,KNN,1.118682


## Voting Regressor (Averaging Predictions)

- The Voting Regressor combines predictions by averaging the outputs of all regressors.

In [20]:
voting_reg = VotingRegressor(
    estimators=[
        ('lr', reg1),
        ('dt', reg2),
        ('knn', reg3)
    ]
)

voting_reg.fit(Xr_train, yr_train)

y_pred_voting = voting_reg.predict(Xr_test)
voting_mse = mean_squared_error(yr_test, y_pred_voting)

print("Voting Regressor MSE:", voting_mse)

Voting Regressor MSE: 0.4584153367514781


- By averaging predictions from multiple regressors, the Voting Regressor reduces individual model errors and produces more stable and accurate predictions.


# Conclusion
- Voting ensembles improve performance by combining predictions from multiple models. In classification, majority voting often increases accuracy, while in regression, averaging predictions reduces error and improves stability.